# Standardized Robustness Pipeline: Evasion (HSJ, Boundary, ZOO)

This notebook runs a standardized robustness pipeline across several models.

## Pipeline steps

1. **Load dataset**
   - Read the CSV, separate features and labels, drop non-feature columns.

2. **Create one canonical split**
   - Make a single train/test split once.
   - The test split is never used for training.

3. **Baseline training + evaluation**
   - Train each model on the clean train set.
   - Evaluate clean accuracy on the held-out test set.

4. **Adversarial training (train-only rounds)**
   - Generate adversarial examples from the training set and append them for robust training.

5. **ART detectors (evasion-time only, optional)**
   - Wrap models/data for ART.
   - Run `art.defences.detector.evasion` detectors on adversarial examples where enabled.

6. **Results table**
   - All metrics are collected into one dataframe (`results_df`) with columns for model, phase, round, attack, and accuracies.

7. **Robustness evaluation under attack**
   - Generate adversarial examples on a subset of the test set using ART attacks (e.g., HSJ, ZOO; Boundary skipped for NN).
   - Record clean accuracy, adversarial accuracy, and accuracy drop.

In [52]:
# ===== Imports =====
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Your model runners (expected to exist in your project)
# NOTE: If running this notebook outside your project root, set PYTHONPATH accordingly.
from MachineLearning.LogReg.LogisticRegression_ML import run_best_model as run_logreg
from MachineLearning.NeuralNetworks.NeuralNet_ML import run_best_model as run_neuralnet
from MachineLearning.RandomForest.RandomForest_ML import run_best_model as run_randomforest
from MachineLearning.SVM.SVM_ML import run_best_model as run_svm
from MachineLearning.XGBoost.XGBoost_ML import run_best_model as run_xgboost

# ART: import SklearnClassifier in a way that avoids importing KerasClassifier
# (some ART installs break if keras-related utils are missing).
try:
    from art.estimators.classification.scikitlearn import SklearnClassifier
except Exception:
    from art.estimators.classification import SklearnClassifier

from art.attacks.evasion import HopSkipJump, BoundaryAttack, ZooAttack

RNG = np.random.default_rng(42)

import warnings
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names",
    category=UserWarning,
 )

In [53]:
# ===== Configuration =====
DATASET_PATH = "CSVs\\dataset.csv"  # change if needed
LABEL_COL = "anomaly"

# Columns often present in your project; will be dropped from features if they exist
DROP_COLS = {"anomaly", "timestamp", "channel", "label", "segment", "train"}

# Train/test split (canonical, used for ALL models & attacks)
TEST_SIZE = 0.7
SPLIT_RANDOM_STATE = 42

# Attack budget knobs (keep realistic)
EVAL_ATTACK_SAMPLES = 50      # number of test points to attack per model/attack
TRAIN_ADV_SAMPLES = 50        # number of train points to adversarially augment per round
ROUNDS = 2                     # adversarial training rounds (Option B)

# Attack configs
HSJ_KWARGS = dict(
    max_iter=15,        # 15 -> 10 (big speedup)
    max_eval=1500,       # 1500 -> 800
    init_eval=25,       # 25 -> 20
    init_size=200,      # 200 -> 100
    targeted=False,
    norm=2
)


HSJ_TRAIN_KWARGS = dict(
    max_iter=8,         # 8 -> 5
    max_eval=1500,       # 300 -> 200
    init_eval=25,
    init_size=200,
    targeted=False,
    norm=2
)

BOUNDARY_KWARGS = dict(
    targeted=False,
    max_iter=100,       # 200 -> 40
    init_size=10         # 10 -> 5
)

ZOO_KWARGS = dict(
    max_iter=5,               # 5 -> 3
    binary_search_steps=1,
    nb_parallel=1,
    batch_size=1
)


# Evasion detectors (run on adversarial examples at test-time)
# Use names like "ART_EVASION::<ClassName>" from art.defences.detector.evasion
EVASION_DETECTORS = [  
    "ART_EVASION::BinaryInputDetector",
    "ART_EVASION::BinaryActivationDetector",
    "ART_EVASION::SubsetScanningDetector",]  # e.g., ["ART_EVASION::BinaryInputDetector"]


In [54]:
# ===== Load dataset and create canonical split =====
assert os.path.exists(DATASET_PATH), f"Dataset not found at: {DATASET_PATH}"

df = pd.read_csv(DATASET_PATH)

# Build feature columns (keep only non-label columns; drop known metadata columns if present)
feature_cols = [c for c in df.columns if c not in DROP_COLS and c != LABEL_COL]

# Keep only numeric features for attacks; if you have categorical columns, encode them before this notebook.
X_all = df[feature_cols].to_numpy(dtype=np.float32)
y_all = df[LABEL_COL].to_numpy(dtype=int)

# Ensure labels are 0..K-1
_, y_all = np.unique(y_all, return_inverse=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=TEST_SIZE, random_state=SPLIT_RANDOM_STATE, stratify=y_all
)

print("Train:", X_train.shape, y_train.shape, "classes:", len(np.unique(y_train)))
print("Test :", X_test.shape, y_test.shape)


Train: (636, 19) (636,) classes: 2
Test : (1487, 19) (1487,)


In [55]:
# ===== Helpers: build train-only CSVs for your existing model runners =====
# Your run_best_model functions read from CSV and do their own internal split.
# To enforce "never train on test", we give them TRAIN-ONLY CSVs.
# We then evaluate the returned trained pipeline on our held-out X_test/y_test.

WORK_DIR = "StandardizedRuns"
os.makedirs(WORK_DIR, exist_ok=True)

def make_train_df_from_arrays(X_tr: np.ndarray, y_tr: np.ndarray) -> pd.DataFrame:
    out = pd.DataFrame(X_tr, columns=feature_cols)
    out[LABEL_COL] = y_tr
    return out

def save_train_csv(df_train: pd.DataFrame, name: str) -> str:
    path = os.path.join(WORK_DIR, name)
    df_train.to_csv(path, index=False)
    return path

def fit_model_with_runner(model_name: str, runner_fn, train_csv_path: str):
    # Use a more reasonable internal split than some module defaults.
    # Many of your modules default to test_size=0.80 (very large). Override to 0.2.
    pipe, _internal_X_test, _internal_y_test = runner_fn(
        path=train_csv_path,
        test_size=0.2,
        random_state=SPLIT_RANDOM_STATE
    )
    return pipe

def eval_clean(pipe, X: np.ndarray, y: np.ndarray) -> float:
    y_pred = pipe.predict(X)
    return float(accuracy_score(y, y_pred))


In [56]:
# ===== ART helpers (no detectors) =====




def wrap_art(pipe, X_ref: np.ndarray) -> SklearnClassifier:
    # clip_values: pragmatic min/max bound from training data
    clip_values = (float(np.min(X_ref)), float(np.max(X_ref)))
    return SklearnClassifier(model=pipe, clip_values=clip_values)

def predict_labels_art(art_clf: SklearnClassifier, X: np.ndarray) -> np.ndarray:
    preds = np.asarray(art_clf.predict(X))
    if preds.ndim == 1:
        return preds.astype(int)
    return np.argmax(preds, axis=1)

def sample_subset(X: np.ndarray, y: np.ndarray, n: int, rng=RNG):
    if n >= len(X):
        return X, y
    idx = rng.choice(len(X), size=n, replace=False)
    return X[idx], y[idx]

def attack_success_rate(y_true: np.ndarray, y_pred_clean: np.ndarray, y_pred_adv: np.ndarray) -> float:
    mask = (y_pred_clean == y_true)
    if mask.sum() == 0:
        return float('nan')
    return float((y_pred_adv[mask] != y_true[mask]).mean())

def eval_under_attack(attack_name: str, art_clf: SklearnClassifier, X_eval: np.ndarray, y_eval: np.ndarray):
    # Generate adversarial examples for evaluation subset and compute metrics.
    if attack_name == "HSJ":
        atk = HopSkipJump(classifier=art_clf, **HSJ_KWARGS)
        X_adv = atk.generate(x=X_eval, y=y_eval)
    elif attack_name == "Boundary":
        # Skip Boundary for NN (sklearn MLP) because BoundaryAttack can produce NaNs
        try:
            m = getattr(art_clf, "model", None)

            # unwrap common wrapper
            if hasattr(m, "base_model"):
                m = m.base_model

            # unwrap sklearn Pipeline -> final estimator
            if hasattr(m, "steps") and len(m.steps) > 0:
                m = m.steps[-1][1]

            if m is not None and m.__class__.__name__ == "MLPClassifier":
                return np.nan, np.nan, np.nan, np.nan, {}
        except Exception:
            pass

        atk = BoundaryAttack(estimator=art_clf, **BOUNDARY_KWARGS)
        X_adv = atk.generate(x=X_eval, y=y_eval)

    elif attack_name == "ZOO":
        atk = ZooAttack(classifier=art_clf, **ZOO_KWARGS)
        # ZOO often expects one-hot labels in some setups, but can work with integers depending on estimator.
        # We'll try integer labels first; if it errors, we fall back to simple one-hot.
        try:
            X_adv = atk.generate(x=X_eval, y=y_eval)
        except Exception:
            k = int(len(np.unique(y_train)))
            y_oh = np.zeros((len(y_eval), k), dtype=np.float32)
            y_oh[np.arange(len(y_eval)), y_eval.astype(int)] = 1.0
            X_adv = atk.generate(x=X_eval, y=y_oh)
    else:
        raise ValueError(f"Unknown attack: {attack_name}")

    X_adv = np.asarray(X_adv, dtype=np.float32)

    y_pred_clean = predict_labels_art(art_clf, X_eval)
    y_pred_adv = predict_labels_art(art_clf, X_adv)


    # Pass art_clf and X_eval to detector runner so detectors that need them can be instantiated
    ev_det_metrics = run_evasion_detectors_on_adv(X_adv, y_pred_adv, art_clf=art_clf, X_ref=X_eval)
    clean_acc = float(accuracy_score(y_eval, y_pred_clean))
    adv_acc = float(accuracy_score(y_eval, y_pred_adv))
    drop = clean_acc - adv_acc
    asr = attack_success_rate(y_eval, y_pred_clean, y_pred_adv)
    return clean_acc, adv_acc, drop, asr, ev_det_metrics


def _try_import_art_detectors():
    """Best-effort import for ART detector modules. Returns (poison_mod, evasion_mod) or (None, None)."""
    try:
        import art  # noqa: F401
        from art.defences.detector import poison as poison_mod
        from art.defences.detector import evasion as evasion_mod
        return poison_mod, evasion_mod
    except Exception:
        return None, None

def list_art_detector_classes():
    """List available class names in art.defences.detector.poison and .evasion (if installed)."""
    poison_mod, evasion_mod = _try_import_art_detectors()
    out = {"poison": [], "evasion": []}
    import inspect
    if poison_mod is not None:
        for name, obj in vars(poison_mod).items():
            if inspect.isclass(obj) and obj.__module__.startswith("art."):
                out["poison"].append(name)
    if evasion_mod is not None:
        for name, obj in vars(evasion_mod).items():
            if inspect.isclass(obj) and obj.__module__.startswith("art."):
                out["evasion"].append(name)
    out["poison"].sort()
    out["evasion"].sort()
    return out

def _instantiate_art_class(mod, class_name: str, kwargs: dict, *args):
    """Instantiate class from an ART module, filtering kwargs to match the constructor signature."""
    import inspect
    cls = getattr(mod, class_name, None)
    if cls is None:
        raise ImportError(f"ART class not found: {class_name}")
    sig = None
    try:
        sig = inspect.signature(cls.__init__)
    except Exception:
        sig = None
    if sig is not None:
        valid = {k: v for k, v in (kwargs or {}).items() if k in sig.parameters}
    else:
        valid = kwargs or {}
    return cls(*args, **valid)

def run_evasion_detectors_on_adv(X_adv: np.ndarray, y_pred_adv: np.ndarray, art_clf=None, X_ref=None):
    """Run configured EVASION_DETECTORS and return dict of metrics.

    This best-effort runner will inspect detector constructors and pass the
    `art_clf` (ART classifier) and/or `X_ref` (reference examples) as positional
    args when the detector's __init__ signature indicates they are expected.
    """
    metrics = {}
    if not EVASION_DETECTORS:
        return metrics

    _, evasion_mod = _try_import_art_detectors()
    if evasion_mod is None:
        for det in EVASION_DETECTORS:
            metrics[f"evasion_det::{det}"] = np.nan
            metrics[f"evasion_det::{det}::error"] = "ART not installed or import failed"
        return metrics

    import numpy as np
    import traceback
    import inspect
    for det_name in EVASION_DETECTORS:
        if not det_name.startswith("ART_EVASION::"):
            continue
        class_name = det_name.split("::", 1)[1]
        kwargs = EVASION_DETECTOR_KWARGS.get(class_name, {}) if "EVASION_DETECTOR_KWARGS" in globals() else {}
        try:
            # Decide positional args to pass based on the detector constructor signature.
            cls = getattr(evasion_mod, class_name, None)
            pos_args = []
            if cls is None:
                raise ImportError(f"ART detector class not found: {class_name}")
            try:
                sig = inspect.signature(cls.__init__)
                param_names = [p.name for p in sig.parameters.values() if p.name != 'self']
            except Exception:
                param_names = []

            # If constructor expects a classifier/estimator, pass art_clf
            if any(n in param_names for n in ("estimator", "classifier", "model")) and art_clf is not None:
                pos_args.append(art_clf)

            # If constructor expects reference data, pass X_ref (fallback to X_adv)
            if any(n in param_names for n in ("x", "x_train", "reference_data", "examples", "X", "X_train", "reference", "x_ref")):
                pos_args.append(X_ref if X_ref is not None else X_adv)

            det = _instantiate_art_class(evasion_mod, class_name, kwargs, *pos_args)
            # Try common APIs
            if hasattr(det, "detect"):
                out = det.detect(X_adv)  # some detectors take only x
            elif hasattr(det, "predict"):
                out = det.predict(X_adv)
            else:
                raise AttributeError("Detector has no detect/predict method")
            out = np.asarray(out).ravel()
            # Interpret: if boolean -> True means detected; if scores -> threshold at 0.5 (best-effort)
            if out.dtype == bool:
                detected = out
            else:
                detected = out > 0.5
            metrics[f"evasion_det::{class_name}::detected_rate"] = float(np.mean(detected))
        except Exception as e:
            # Capture exception for debugging so NaNs can be investigated
            tb = traceback.format_exc()
            metrics[f"evasion_det::{class_name}::detected_rate"] = np.nan
            metrics[f"evasion_det::{class_name}::error"] = str(e)
            metrics[f"evasion_det::{class_name}::traceback"] = tb
    return metrics


In [57]:
# ===== Standardized experiment runners =====

MODEL_RUNNERS = {
    "LogReg": run_logreg,
    "NeuralNet": run_neuralnet,
    "RandomForest": run_randomforest,
    "SVM": run_svm,
    # "XGBoost": run_xgboost,
}

EVASION_ATTACKS = ["HSJ", "Boundary", "ZOO"]

def _flatten_metrics(d: dict, prefix: str = ""):
    if not d:
        return {}
    return {f"{prefix}{k}": v for k, v in d.items()}


def run_baseline_and_evasion(model_name: str, runner_fn):
    rows = []

    # Train on clean TRAIN ONLY
    train_df = make_train_df_from_arrays(X_train, y_train)
    clean_train_path = save_train_csv(train_df, f"{model_name}_train_clean.csv")
    pipe = fit_model_with_runner(model_name, runner_fn, clean_train_path)

    # Clean eval (full held-out test)
    clean_test_acc = eval_clean(pipe, X_test, y_test)

    # Evasion eval on subset
    art_clf = wrap_art(pipe, X_train)
    X_eval, y_eval = sample_subset(X_test, y_test, EVAL_ATTACK_SAMPLES)

    for atk in EVASION_ATTACKS:
        cacc, aacc, drop, asr, evm = eval_under_attack(atk, art_clf, X_eval, y_eval)
        row = {
            "model": model_name,
            "phase": "baseline",
            "attack_type": "evasion",
            "attack": atk,
            "round": 0,
            "clean_test_acc": clean_test_acc,
            "clean_acc_evalsubset": cacc,
            "adv_acc_evalsubset": aacc,
            "acc_drop_evalsubset": drop,
            "attack_success_rate": asr,
            "train_poison_rate": 0.0,
            "train_adv_augmented": 0,
            "eval_attack_samples": len(X_eval),
        }
        # Merge any evasion-detector metrics into the result row
        row.update(_flatten_metrics(evm))
        rows.append(row)

    return pipe, pd.DataFrame(rows)

def run_adversarial_training(model_name: str, runner_fn, train_attack: str = "HSJ"):
    rows = []

    # Start with clean train
    X_tr = X_train.copy()
    y_tr = y_train.copy()
    augmented = 0

    for r in range(ROUNDS + 1):
        # Train current model on current train set (clean + accumulated adv)
        train_df = make_train_df_from_arrays(X_tr, y_tr)
        train_path = save_train_csv(train_df, f"{model_name}_train_advtrain_round{r}.csv")
        pipe = fit_model_with_runner(model_name, runner_fn, train_path)

        # Evaluate on full clean test
        clean_test_acc = eval_clean(pipe, X_test, y_test)

        # Evaluate evasion robustness on subset (HSJ/Boundary/ZOO)
        art_clf = wrap_art(pipe, X_tr)
        X_eval, y_eval = sample_subset(X_test, y_test, EVAL_ATTACK_SAMPLES)

        for atk in EVASION_ATTACKS:
            cacc, aacc, drop, asr, evm = eval_under_attack(atk, art_clf, X_eval, y_eval)
            row = {
                "model": model_name,
                "phase": "advtrain",
                "attack_type": "evasion",
                "attack": atk,
                "round": r,
                "clean_test_acc": clean_test_acc,
                "clean_acc_evalsubset": cacc,
                "adv_acc_evalsubset": aacc,
                "acc_drop_evalsubset": drop,
                "attack_success_rate": asr,
                "train_poison_rate": 0.0,
                "train_adv_augmented": augmented,
                "eval_attack_samples": len(X_eval),
            }
            # Merge detector metrics into the row
            row.update(_flatten_metrics(evm))
            rows.append(row)

        if r == ROUNDS:
            break

        # Generate fresh adversarials from TRAIN subset only
        X_sub, y_sub = sample_subset(X_tr, y_tr, TRAIN_ADV_SAMPLES)

        if train_attack == "HSJ":
            atk_train = HopSkipJump(classifier=art_clf, **HSJ_TRAIN_KWARGS)
            X_adv = atk_train.generate(x=X_sub, y=y_sub)
        elif train_attack == "Boundary":
            atk_train = BoundaryAttack(estimator=art_clf, **BOUNDARY_KWARGS)
            X_adv = atk_train.generate(x=X_sub, y=y_sub)
        elif train_attack == "ZOO":
            atk_train = ZooAttack(classifier=art_clf, **ZOO_KWARGS)
            try:
                X_adv = atk_train.generate(x=X_sub, y=y_sub)
            except Exception:
                k = int(len(np.unique(y_train)))
                y_oh = np.zeros((len(y_sub), k), dtype=np.float32)
                y_oh[np.arange(len(y_sub)), y_sub.astype(int)] = 1.0
                X_adv = atk_train.generate(x=X_sub, y=y_oh)
        else:
            raise ValueError(train_attack)

        X_adv = np.asarray(X_adv, dtype=np.float32)

        # Append with correct labels
        X_tr = np.vstack([X_tr, X_adv]).astype(np.float32)
        y_tr = np.concatenate([y_tr, y_sub]).astype(int)
        augmented += len(X_adv)

        print(f"[{model_name}] adv-train round {r} -> {r+1}: +{len(X_adv)} using {train_attack}, train size={len(X_tr)}")

    return pd.DataFrame(rows)

In [58]:
# ===== Run the full standardized suite =====

all_rows = []

for model_name, runner_fn in MODEL_RUNNERS.items():
    print("\n" + "="*80)
    print("MODEL:", model_name)
    print("="*80)

    # Baseline + evasion attacks (clean training)
    _pipe, df_base = run_baseline_and_evasion(model_name, runner_fn)
    all_rows.append(df_base)

    # Option B: adversarial training (default HSJ) - comment out if too slow
    df_advtrain = run_adversarial_training(model_name, runner_fn, train_attack="HSJ")
    all_rows.append(df_advtrain)

results_df = pd.concat(all_rows, ignore_index=True)
results_df


MODEL: LogReg
Using parameters: {'clf__C': 10, 'clf__class_weight': 'balanced', 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}

Saved last run results to Results/LogRegResults\results_logreg.csv
Saved ROC data to Results/LogRegResults\roc_logreg_clean.csv (AUC = 0.995)
Saved summary (AUC + Confusion Matrix) to Results/LogRegResults\logreg_summary.csv

=== Test Set Classification Report ===
              precision    recall  f1-score   support

           0       0.99      0.93      0.96       102
           1       0.78      0.96      0.86        26

    accuracy                           0.94       128
   macro avg       0.89      0.95      0.91       128
weighted avg       0.95      0.94      0.94       128


Confusion Matrix:
         Pred 0  Pred 1
True 0      95       7
True 1       1      25

AUC: 0.995

=== All results and summaries saved successfully ===


ZOO: 100%|██████████| 50/50 [00:00<00:00, 369.08it/s]


Using parameters: {'clf__C': 10, 'clf__class_weight': 'balanced', 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}

Saved last run results to Results/LogRegResults\results_logreg.csv
Saved ROC data to Results/LogRegResults\roc_logreg_clean.csv (AUC = 0.995)
Saved summary (AUC + Confusion Matrix) to Results/LogRegResults\logreg_summary.csv

=== Test Set Classification Report ===
              precision    recall  f1-score   support

           0       0.99      0.93      0.96       102
           1       0.78      0.96      0.86        26

    accuracy                           0.94       128
   macro avg       0.89      0.95      0.91       128
weighted avg       0.95      0.94      0.94       128


Confusion Matrix:
         Pred 0  Pred 1
True 0      95       7
True 1       1      25

AUC: 0.995

=== All results and summaries saved successfully ===


HopSkipJump: 100%|██████████| 50/50 [00:00<00:00, 56.44it/s]


[LogReg] adv-train round 0 -> 1: +50 using HSJ, train size=686
Using parameters: {'clf__C': 10, 'clf__class_weight': 'balanced', 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}

Saved last run results to Results/LogRegResults\results_logreg.csv
Saved ROC data to Results/LogRegResults\roc_logreg_clean.csv (AUC = 0.978)
Saved summary (AUC + Confusion Matrix) to Results/LogRegResults\logreg_summary.csv

=== Test Set Classification Report ===
              precision    recall  f1-score   support

           0       0.98      0.91      0.94       109
           1       0.73      0.93      0.82        29

    accuracy                           0.91       138
   macro avg       0.85      0.92      0.88       138
weighted avg       0.93      0.91      0.92       138


Confusion Matrix:
         Pred 0  Pred 1
True 0      99      10
True 1       2      27

AUC: 0.978

=== All results and summaries saved successfully ===


...

[SVM] adv-train round 0 -> 1: +50 using HSJ, train size=686

Saved last run results to Results/SVMResults\results_svm.csv
Saved ROC data to Results/SVMResults\roc_svm_clean.csv (AUC = 0.913)
Saved summary (AUC + Confusion Matrix) to Results/SVMResults\svm_summary.csv

=== SVM Test Set Report ===
              precision    recall  f1-score   support

           0       0.93      0.97      0.95       493
           1       0.85      0.71      0.77       125

    accuracy                           0.92       618
   macro avg       0.89      0.84      0.86       618
weighted avg       0.91      0.92      0.91       618


Confusion Matrix:
         Pred 0  Pred 1
True 0     477      16
True 1      36      89

AUC: 0.913

=== All results and summaries saved successfully ===


HopSkipJump: 100%|██████████| 50/50 [00:00<00:00, 55.91it/s]


[SVM] adv-train round 1 -> 2: +50 using HSJ, train size=736

Saved last run results to Results/SVMResults\results_svm.csv
Saved ROC data to Results/SVMResults\roc_svm_clean.csv (AUC = 0.937)
Saved summary (AUC + Confusion Matrix) to Results/SVMResults\svm_summary.csv

=== SVM Test Set Report ===
              precision    recall  f1-score   support

           0       0.92      0.97      0.95       530
           1       0.86      0.66      0.75       133

    accuracy                           0.91       663
   macro avg       0.89      0.82      0.85       663
weighted avg       0.91      0.91      0.91       663


Confusion Matrix:
         Pred 0  Pred 1
True 0     516      14
True 1      45      88

AUC: 0.937

=== All results and summaries saved successfully ===


ZOO: 100%|██████████| 50/50 [00:00<00:00, 380.18it/s]


,model,phase,attack_type,attack,round,clean_test_acc,clean_acc_evalsubset,adv_acc_evalsubset,acc_drop_evalsubset,attack_success_rate,...,eval_attack_samples,evasion_det::BinaryInputDetector::detected_rate,evasion_det::BinaryInputDetector::error,evasion_det::BinaryInputDetector::traceback,evasion_det::BinaryActivationDetector::detected_rate,evasion_det::BinaryActivationDetector::error,evasion_det::BinaryActivationDetector::traceback,evasion_det::SubsetScanningDetector::detected_rate,evasion_det::SubsetScanningDetector::error,evasion_det::SubsetScanningDetector::traceback
0,LogReg,baseline,evasion,HSJ,0,0.919301,0.98,0.02,0.96,1.000000,...,50,NaN,BinaryInputDetector.__init__() missing 1 requi...,"Traceback (most recent call last):\n File ""C:...",NaN,BinaryActivationDetector.__init__() missing 2 ...,"Traceback (most recent call last):\n File ""C:...",NaN,SubsetScanningDetector.__init__() missing 2 re...,"Traceback (most recent call last):\n File ""C:..."
1,LogReg,baseline,evasion,Boundary,0,0.919301,0.98,0.52,0.46,0.489796,...,50,NaN,BinaryInputDetector.__init__() missing 1 requi...,"Traceback (most recent call last):\n File ""C:...",NaN,BinaryActivationDetector.__init__() missing 2 ...,"Traceback (most recent call last):\n File ""C:...",NaN,SubsetScanningDetector.__init__() missing 2 re...,"Traceback (most recent call last):\n File ""C:..."
2,LogReg,baseline,evasion,ZOO,0,0.919301,0.98,0.38,0.60,0.612245,...,50,NaN,BinaryInputDetector.__init__() missing 1 requi...,"Traceback (most recent call last):\n File ""C:...",NaN,BinaryActivationDetector.__init__() missing 2 ...,"Traceback (most recent call last):\n File ""C:...",NaN,SubsetScanningDetector.__init__() missing 2 re...,"Traceback (most recent call last):\n File ""C:..."
3,LogReg,advtrain,evasion,HSJ,0,0.919301,0.90,0.10,0.80,1.000000,...,50,NaN,BinaryInputDetector.__init__() missing 1 requi...,"Traceback (most recent call last):\n File ""C:...",NaN,BinaryActivationDetector.__init__() missing 2 ...,"Traceback (most recent call last):\n File ""C:...",NaN,SubsetScanningDetector.__init__() missing 2 re...,"Traceback (most recent call last):\n File ""C:..."
4,LogReg,advtrain,evasion,Boundary,0,0.919301,0.90,0.40,0.50,0.600000,...,50,NaN,BinaryInputDetector.__init__() missing 1 requi...,"Traceback (most recent call last):\n File ""C:...",NaN,BinaryActivationDetector.__init__() missing 2 ...,"Traceback (most recent call last):\n File ""C:...",NaN,SubsetScanningDetector.__init__() missing 2 re...,"Traceback (most recent call last):\n File ""C:..."
5,LogReg,advtrain,evasion,ZOO,0,0.919301,0.90,0.26,0.64,0.711111,...,50,NaN,BinaryInputDetector.__init__() missing 1 requi...,"Traceback (most recent call last):\n File ""C:...",NaN,BinaryActivationDetector.__init__() missing 2 ...,"Traceback (most recent call last):\n File ""C:...",NaN,SubsetScanningDetector.__init__() missing 2 re...,"Traceback (most recent call last):\n File ""C:..."
6,LogReg,advtrain,evasion,HSJ,1,0.915938,0.96,0.04,0.92,1.000000,...,50,NaN,BinaryInputDetector.__init__() missing 1 requi...,"Traceback (most recent call last):\n File ""C:...",NaN,BinaryActivationDetector.__init__() missing 2 ...,"Traceback (most recent call last):\n File ""C:...",NaN,SubsetScanningDetector.__init__() missing 2 re...,"Traceback (most recent call last):\n File ""C:..."
7,LogReg,advtrain,evasion,Boundary,1,0.915938,0.96,0.18,0.78,0.854167,...,50,NaN,BinaryInputDetector.__init__() missing 1 requi...,"Traceback (most recent call last):\n File ""C:...",NaN,BinaryActivationDetector.__init__() missing 2 ...,"Traceback (most recent call last):\n File ""C:...",NaN,SubsetScanningDetector.__init__() missing 2 re...,"Traceback (most recent call last):\n File ""C:..."
8,LogReg,advtrain,evasion,ZOO,1,0.915938,0.96,0.42,0.54,0.562500,...,50,NaN,BinaryInputDetector.__init__() missing 1 requi...,"Traceback (most recent call last):\n File ""C:...",NaN,BinaryActivationDetector.__init__() missing 2 ...,"Traceback (most recent call last):\n File ""C:...",NaN,

In [59]:
# ===== Save results =====
out_csv = os.path.join(WORK_DIR, "standardized_results.csv")
results_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)

# Only baseline + adversarial training
df = results_df[results_df["phase"].isin(["baseline", "advtrain"])]

# --- Clean accuracy (no attack dimension) ---
clean_acc = (
    df.groupby(["model", "phase", "round"])["clean_test_acc"]
      .mean()
)

# --- Attack-dependent metrics ---
attack_metrics = df.pivot_table(
    index=["model", "phase", "round"],
    columns="attack",
    values=[
        "adv_acc_evalsubset",
        "acc_drop_evalsubset",
        "attack_success_rate"
    ],
    aggfunc="mean"
)


# --- Combine ---
pivot = pd.concat([clean_acc, attack_metrics], axis=1)

# Round for readability
pivot = pivot.round(3)


pivot


Saved: StandardizedRuns\standardized_results.csv


clean_test_acc  (acc_drop_evalsubset, Boundary)  \
model        phase    round                                                    
LogReg       advtrain 0               0.919                             0.50   
                      1               0.916                             0.78   
                      2               0.924                             0.80   
             baseline 0               0.919                             0.46   
NeuralNet    advtrain 0               0.941                              NaN   
                      1               0.940                              NaN   
                      2               0.954                              NaN   
             baseline 0               0.941                              NaN   
RandomForest advtrain 0               0.954                             0.22   
                      1               0.951                             0.18   
                      2               0.948                             0.14   
             baseline 0               0.954                             0.20   
SVM          advtrain 0               0.884                             0.62   
                      1               0.917                             0.86   
                      2               0.912                             0.80   
             baseline 0               0.884                             0.68   

                             (acc_drop_evalsubset, HSJ)  \
model        phase    round                               
LogReg       advtrain 0                            0.80   
                      1                            0.92   
                      2                            0.84   
             baseline 0                            0.96   
NeuralNet    advtrain 0                            0.76   
                      1                            0.76   
                      2                            0.80   
             baseline 0                            0.92   
RandomForest advtrain 0                            0.68   
                      1                            0.18   
                      2                            0.38   
             baseline 0                            0.70   
SVM          advtrain 0                            0.60   
                      1                            0.88   
                      2                            0.80   
             baseline 0                            0.72   

                             (acc_drop_evalsubset, ZOO)  \
model        phase    round                               
LogReg       advtrain 0                            0.64   
                      1                            0.54   
                      2                            0.62   
             baseline 0                            0.60   
NeuralNet    advtrain 0                            0.42   
                      1                            0.50   
                      2                            0.42   
             baseline 0                            0.48   
RandomForest advtrain 0                            0.00   
                      1                            0.00   
                      2                            0.04   
             baseline 0                            0.00   
SVM          advtrain 0                            0.42   
                      1                            0.56   
                      2                            0.00   
             baseline 0                            0.32   

                             (adv_acc_evalsubset, Boundary)  \
model        phase    round                                   
LogReg       advtrain 0                                0.40   
                      1                                0.18   
                      2                                0.12   
             baseline 0                                0.52   
NeuralNet    advtrain 0                                 NaN   
                     

In [60]:
# ===== Detector sanity check & inspection =====

# 1. List all detector-related columns
detector_cols = [c for c in results_df.columns if "det" in c.lower()]

print("Detector columns found:")
for c in detector_cols:
    print("  ", c)

if not detector_cols:
    print("⚠️  No detector columns found. Detectors are NOT running.")

# 2. Show a compact table of detector outputs
cols_to_show = (
    ["model", "phase", "round", "attack"] +
    detector_cols
)

display(
    results_df[cols_to_show]
        .sort_values(["model", "phase", "round", "attack"])
        .round(3)
)

# 3. Check if detectors are constant or meaningful
print("\nDetector value distributions:")
for c in detector_cols:
    print(f"\n{c}")
    print(results_df[c].value_counts(dropna=False))

# 4. Mean detector response per attack (sanity check)
print("\nMean detector response per attack:")
for c in detector_cols:
    print(f"\n{c}")
    print(
        results_df
            .groupby("attack")[c]
            .mean()
            .round(3)
    )


Detector columns found:
   evasion_det::BinaryInputDetector::detected_rate
   evasion_det::BinaryInputDetector::error
   evasion_det::BinaryInputDetector::traceback
   evasion_det::BinaryActivationDetector::detected_rate
   evasion_det::BinaryActivationDetector::error
   evasion_det::BinaryActivationDetector::traceback
   evasion_det::SubsetScanningDetector::detected_rate
   evasion_det::SubsetScanningDetector::error
   evasion_det::SubsetScanningDetector::traceback


,model,phase,round,attack,evasion_det::BinaryInputDetector::detected_rate,evasion_det::BinaryInputDetector::error,evasion_det::BinaryInputDetector::traceback,evasion_det::BinaryActivationDetector::detected_rate,evasion_det::BinaryActivationDetector::error,evasion_det::BinaryActivationDetector::traceback,evasion_det::SubsetScanningDetector::detected_rate,evasion_det::SubsetScanningDetector::error,evasion_det::SubsetScanningDetector::traceback
4,LogReg,advtrain,0,Boundary,NaN,BinaryInputDetector.__init__() missing 1 requi...,"Traceback (most recent call last):\n File ""C:...",NaN,BinaryActivationDetector.__init__() missing 2 ...,"Traceback (most recent call last):\n File ""C:...",NaN,SubsetScanningDetector.__init__() missing 2 re...,"Traceback (most recent call last):\n File ""C:..."
3,LogReg,advtrain,0,HSJ,NaN,BinaryInputDetector.__init__() missing 1 requi...,"Traceback (most recent call last):\n File ""C:...",NaN,BinaryActivationDetector.__init__() missing 2 ...,"Traceback (most recent call last):\n File ""C:...",NaN,SubsetScanningDetector.__init__() missing 2 re...,"Traceback (most recent call last):\n File ""C:..."
5,LogReg,advtrain,0,ZOO,NaN,BinaryInputDetector.__init__() missing 1 requi...,"Traceback (most recent call last):\n File ""C:...",NaN,BinaryActivationDetector.__init__() missing 2 ...,"Traceback (most recent call last):\n File ""C:...",NaN,SubsetScanningDetector.__init__() missing 2 re...,"Traceback (most recent call last):\n File ""C:..."
7,LogReg,advtrain,1,Boundary,NaN,BinaryInputDetector.__init__() missing 1 requi...,"Traceback (most recent call last):\n File ""C:...",NaN,BinaryActivationDetector.__init__() missing 2 ...,"Traceback (most recent call last):\n File ""C:...",NaN,SubsetScanningDetector.__init__() missing 2 re...,"Traceback (most recent call last):\n File ""C:..."
6,LogReg,advtrain,1,HSJ,NaN,BinaryInputDetector.__init__() missing 1 requi...,"Traceback (most recent call last):\n File ""C:...",NaN,BinaryActivationDetector.__init__() missing 2 ...,"Traceback (most recent call last):\n File ""C:...",NaN,SubsetScanningDetector.__init__() missing 2 re...,"Traceback (most recent call last):\n File ""C:..."
8,LogReg,advtrain,1,ZOO,NaN,BinaryInputDetector.__init__() missing 1 requi...,"Traceback (most recent call last):\n File ""C:...",NaN,BinaryActivationDetector.__init__() missing 2 ...,"Traceback (most recent call last):\n File ""C:...",NaN,SubsetScanningDetector.__init__() missing 2 re...,"Traceback (most recent call last):\n File ""C:..."
10,LogReg,advtrain,2,Boundary,NaN,BinaryInputDetector.__init__() missing 1 requi...,"Traceback (most recent call last):\n File ""C:...",NaN,BinaryActivationDetector.__init__() missing 2 ...,"Traceback (most recent call last):\n File ""C:...",NaN,SubsetScanningDetector.__init__() missing 2 re...,"Traceback (most recent call last):\n File ""C:..."
9,LogReg,advtrain,2,HSJ,NaN,BinaryInputDetector.__init__() missing 1 requi...,"Traceback (most recent call last):\n File ""C:...",NaN,BinaryActivationDetector.__init__() missing 2 ...,"Traceback (most recent call last):\n File ""C:...",NaN,SubsetScanningDetector.__init__() missing 2 re...,"Traceback (most recent call last):\n File ""C:..."
11,LogReg,advtrain,2,ZOO,NaN,BinaryInputDetector.__init__() missing 1 requi...,"Traceback (most recent call last):\n File ""C:...",NaN,BinaryActivationDetector.__init__() missing 2 ...,"Traceback (most recent call last):\n File ""C:...",NaN,SubsetScanningDetector.__init__() missing 2 re...,"Traceback (most recent call last):\n File ""C:..."
1,LogReg,baseline,0,Boundary,NaN,BinaryInputDetector.__init__() missing 1 requi...,"Traceback (most recent call last):\n File ""C:...",NaN,BinaryActivationDetector.__init__() missing 2 ...,"Traceback (most recent call last):\n File ""C:...",NaN,SubsetScanningDetector.__init__() missing 2 re...,"Traceback (most recent call last):\n File ""C:..."



Detector value distributions:

evasion_det::BinaryInputDetector::detected_rate
evasion_det::BinaryInputDetector::detected_rate
NaN    48
Name: count, dtype: int64

evasion_det::BinaryInputDetector::error
evasion_det::BinaryInputDetector::error
BinaryInputDetector.__init__() missing 1 required positional argument: 'detector'    44
NaN                                                                                   4
Name: count, dtype: int64

evasion_det::BinaryInputDetector::traceback
evasion_det::BinaryInputDetector::traceback
Traceback (most recent call last):\n  File "C:\Users\Jan\AppData\Local\Temp\ipykernel_17832\4110318178.py", line 174, in run_evasion_detectors_on_adv\n    det = _instantiate_art_class(evasion_mod, class_name, kwargs, *pos_args)\n          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File "C:\Users\Jan\AppData\Local\Temp\ipykernel_17832\4110318178.py", line 126, in _instantiate_art_class\n    return cls(*args, **valid)\n           ^^^^^

TypeError: agg function failed [how->mean,dtype->object]